# Evaluating VabamorfTagger and VabamorfWithBertTagger on Estonian Universal Dependencies' EDT corpus

Code is replicated from Sander Saska's code (https://github.com/estnltk/estnltk-model-training/blob/main/morph_tagging/02_eval_UD_Est-EDT_treebank.ipynb) with slight modifications.

It appears that when lemma is included in evaluation, most of the low scores are the result of verb lemma format, where UD has removed the -ma ending while Vabamorf and VabamorfWithBertTagger keep the -ma ending in the lemma. 

Because of this, the last 2 characters of any vabamorf verb lemma are removed. This results the metrics to improve from 0.71 to 0.85.

In [1]:
import os, os.path
import pandas as pd
import estnltk
from estnltk import Text
from estnltk.converters import text_to_json
from est_ud_utils import load_ud_file_texts_with_corrections
from est_ud_morph_conv import convert_ud_layer_to_reduced_morph_layer
from vabamorfwithberttagger_v2 import VabamorfWithBertTagger
from estnltk.taggers import VabamorfTagger
import evaluate

/home/kaire/anaconda3/envs/base2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [90]:
print("estnltk:", estnltk.__version__)
print("evaluate:", evaluate.__version__)
print("pandas:", pd.__version__)

estnltk: 1.7.4
evaluate: 0.4.6
pandas: 2.2.2


## Converting UD treebank to vabamorf format

In [2]:
def convert_ud_to_vabamorf(ud_corpus_dir, output_dir):
    """Converts Universal Dependencies' (UD) corpus to Vabamorf format

    Args:
        ud_corpus_dir (str): path to directory containing UD corpus .conllu files
        output_dir (str): path to directory, where Vabamorf jsons files will be written
    """
    # Create directory if it doesn't exist
    if not os.path.isdir( output_dir ):
        os.makedirs(output_dir)
    assert os.path.isdir( output_dir )

    # Load UD corpus' files as EstNLTK Text objects
    loaded_texts  = []
    ud_layer_name = 'ud_syntax'
    for fname in os.listdir( ud_corpus_dir ):
        #if 'train' in fname:
        #    continue
        #if 'dev' in fname:
        #    continue
        #if 'test' in fname:
        #    continue
        if fname.endswith('.conllu'):
            fpath = os.path.join( ud_corpus_dir, fname )
            texts = load_ud_file_texts_with_corrections( fpath, ud_layer_name )
            for text in texts:
                text.meta['file'] = fname
                loaded_texts.append( text )

    # Convert UD's morphosyntactic annotations to Vabamorf-like annotations
    for tid, text in enumerate(loaded_texts):
        convert_ud_layer_to_reduced_morph_layer( text, 'ud_syntax', 'ud_morph_reduced', add_layer=True )
        fname = text.meta['file'].replace('.conllu', '_'+('{:03d}'.format(tid))+'.json')
        fpath = os.path.join(output_dir, fname)
        estnltk.converters.text_to_json(text, file=fpath)

In [3]:
ud_corpus_dir = "UD_Estonian-EDT-r2.18" # UD Corpus location
output_dir = 'UD_converted' # Output directory

In [4]:
if not os.path.exists(output_dir):
    print("converting")
    convert_ud_to_vabamorf(ud_corpus_dir, output_dir)

converting


## Tagging treebank

In [2]:
vmt = VabamorfTagger()
#vbtf = VabamorfWithBertTagger(use_postanalysis=False)
vbtt = VabamorfWithBertTagger(use_postanalysis=True)

In [3]:
def create_df_ud_corpus(jsons, in_dir, tokenizer, csv_filename, use_postanalysis=None):
    """
    Creates a new dataset from converted the Estonian UD EDT <a href="https://github.com/UniversalDependencies/UD_Estonian-EDT">corpus</a>. \n
    For each <code>.json</code> file, the following info is gathered:
    <ul>
        <li><code>sentence_id</code> -- given for each sentence</li>
        <li><code>words</code> -- words gathered from text</li>
        <li><code>form</code> -- word form notation</li>
        <li><code>pos</code> -- part of speech</li>
        <li><code>file_prefix</code> -- metadata</li>
        <li><code>source</code> -- file name where the text is taken from</li>
    </ul>
    <a href="https://github.com/Filosoft/vabamorf/blob/e6d42371006710175f7ec328c98f90b122930555/doc/tagset.md">Tables of morphological categories</a> for more information about <code>form</code> and <code>pos</code>.

    Args:
        jsons (list[str]): List of json files from which to read in the text
        in_dir (str): Directory containing list of files (<code>jsons</code>)
        tokenizer (str): Use goldstandard (<code>ud_morph_reduced</code>) or Vabamorf tokenization ((<code>morph_analysis</code>))
        csv_filename (str): CSV filename where to save the gathered text
    """
    if tokenizer not in {'ud_morph_reduced', 'morph_analysis', 'custom', 'vabamorftagger'}:
        raise ValueError("create_df_ud_corpus: tokenizer must be one of %r." % {'ud_morph_reduced', 'morph_analysis'})

    tokens = list()
    sentence_id = 0
    fieldnames = ['sentence_id',  'lemma', 'words', 'form', 'pos', 'file_prefix', 'source']

    print("Beginning to morphologically tag file by file. This can take a while.")
    for file_name in jsons:
        # print(f"Beginning to morphologically tag {file_name}")
        sentence_id = 0

        # Morph. tagging
        text = estnltk.converters.json_to_text(file=os.path.join(in_dir, file_name))
        if tokenizer == 'morph_analysis':
            text.tag_layer('morph_analysis')
        
        if tokenizer == 'vabamorftagger':
            text.tag_layer(['words','sentences'])
            vmt.tag( text )
            
        if tokenizer == 'custom':
            if not use_postanalysis:
                vbtf.tag( text )
            elif use_postanalysis:
                vbtt.tag( text )
            
            
        file_prefix = text.meta.get('file_prefix')
        
        for sentence in text.sentences:
            
            if tokenizer == 'ud_morph_reduced':
                sentence_analysis = sentence.ud_morph_reduced
                for text, lemma, form, pos in zip(sentence_analysis.text, sentence_analysis.lemma, sentence_analysis.form, sentence_analysis.pos):
                    if text:
                        tokens.append((sentence_id, lemma[0], text, form[0], pos[0], file_prefix, file_name)) # In case of multiplicity, select the first or index 0
            
            else:
                sentence_analysis = sentence.morph_analysis
                for text, lemma, form, pos in zip(sentence_analysis.text, sentence_analysis.lemma, sentence_analysis.form, sentence_analysis.partofspeech):
                    if text:
                        if pos[0] == "V":
                            lemma = lemma[0][:-2]
                        else:
                            lemma = lemma[0]
                        tokens.append((sentence_id, lemma, text, form[0], pos[0], file_prefix, file_name)) # In case of multiplicity, select the first or index 0
            
            sentence_id += 1
        # print(f"{file_name} tagged")

    print("Morphological tagging completed successfully")
    print("Creating Pandas dataframe")
    df = pd.DataFrame(data=tokens, columns=fieldnames)
    df.to_csv(path_or_buf=csv_filename, index=False)
    print(f"Tagged texts saved to {csv_filename}\n")

In [4]:
def clean_df(df, df_file_name=None):
    """Finishes dataframe by:
    <ul>
        <li>filling NaN values in columns <code>form</code> and <code>pos</code> with empty strings;</li>
        <li>removing NaN words.</li>
    </ul>

    Args:
        df (pandas.core.frame.DataFrame): Pandas dataframe to clean
        df_file_name (str): CSV file name from which dataframe was created
    """
    print("Assigning NaN values in columns form and pos with an empty string")
    # NaN values are assigned with an empty string
    df['lemma'] = df['lemma'].fillna('')
    df['form'] = df['form'].fillna('')
    df['pos'] = df['pos'].fillna('')
    print("Removing NaN words")
    # Removing NaN words
    df.dropna(subset=['words'], inplace=True)
    if df_file_name:
        df.to_csv(path_or_buf=df_file_name, index=False)
        print(f"Modified dataframe saved to {df_file_name}")
    else:
        print("Dataframe cleaned")

In [5]:
def create_labels_column(df, df_file_name=None):
    """
    Creates a new column <code>labels</code> concatenating the values of columns <code>pos</code> (part of speech) and <code>form</code> (word form notation)

    Args:
        df (pandas.core.frame.DataFrame): Pandas dataframe to create a new column
        df_file_name (str): CSV file name from which dataframe was created
    """
    print("Creating column 'labels'")
    df['labels'] = df.apply(lambda row: str(row['form']) + '_' + str(row['pos']) if row['form'] and row['pos'] else str(row['form']) or str(row['pos']), axis=1)
    print("Column 'labels' created")
    if df_file_name:
        df.to_csv(path_or_buf=df_file_name, index=False)
        print(f"Modified dataframe saved to {df_file_name}")
        
        
def create_labels_column2(df, df_file_name=None):
    """
    Creates a new column <code>labels2</code> concatenating the values of columns <code>pos</code> (part of speech), <code>form</code> (word form notation) and <code>lemma</code> (word lemma notation)

    Args:
        df (pandas.core.frame.DataFrame): Pandas dataframe to create a new column
        df_file_name (str): CSV file name from which dataframe was created
    """
    print("Creating column 'labels2'")
    df['labels2'] = df.apply(lambda row: str(row['form']) + '_' + str(row['pos']) + '_' + str(row['lemma']) if row['lemma'] and row['form'] and row['pos'] else str(row['form']) or str(row['pos']) or str(row['lemma']), axis=1)
    print("Column 'labels2' created")
    if df_file_name:
        df.to_csv(path_or_buf=df_file_name, index=False)
        print(f"Modified dataframe saved to {df_file_name}")

In [6]:
ud_dir = "UD_converted"
jsons = os.listdir(ud_dir)

### Baseline

In [8]:
gold = 'ud_andmestik.csv'

if not os.path.exists(gold):
    create_df_ud_corpus(jsons, ud_dir, 'ud_morph_reduced', gold)

Beginning to morphologically tag file by file. This can take a while.
Morphological tagging completed successfully
Creating Pandas dataframe
Tagged texts saved to ud_andmestik.csv



In [9]:
df_ud = pd.read_csv(gold, keep_default_na=False)
clean_df(df_ud, gold)
create_labels_column(df_ud, gold)
create_labels_column2(df_ud, gold.replace(".csv", "2.csv"))

Assigning NaN values in columns form and pos with an empty string
Removing NaN words
Modified dataframe saved to ud_andmestik.csv
Creating column 'labels'
Column 'labels' created
Modified dataframe saved to ud_andmestik.csv
Creating column 'labels2'
Column 'labels2' created
Modified dataframe saved to ud_andmestik2.csv


In [24]:
display(df_ud.head(5))


,sentence_id,lemma,words,form,pos,file_prefix,source,labels,labels2
0,0,Mihhailov,Mihhailovile,sg all,H,aja_ee200110,et_edt-ud-train_007.json,sg all_H,sg all_H_Mihhailov
1,0,määra,määrati,ti,V,aja_ee200110,et_edt-ud-train_007.json,ti_V,ti_V_määra
2,0,preemia,preemia,sg n,S,aja_ee200110,et_edt-ud-train_007.json,sg n_S,sg n_S_preemia
3,0,kapitalism,kapitalismi,sg g,S,aja_ee200110,et_edt-ud-train_007.json,sg g_S,sg g_S_kapitalism
4,0,tingimus,tingimustes,pl in,S,aja_ee200110,et_edt-ud-train_007.json,pl in_S,pl in_S_tingimus


In [28]:
print(df_ud.shape)

(437758, 8)


### VabamorfTagger

In [7]:
vmt_file = 'ud_vabamorftagger.csv'

if not os.path.exists(vmt_file):
    create_df_ud_corpus(jsons, ud_dir, 'vabamorftagger', vmt_file)
    
df_vmt = pd.read_csv(vmt_file, keep_default_na=False)
clean_df(df_vmt, vmt_file)
create_labels_column(df_vmt, vmt_file)
create_labels_column2(df_vmt, vmt_file.replace(".csv", "2.csv"))

Beginning to morphologically tag file by file. This can take a while.
Morphological tagging completed successfully
Creating Pandas dataframe
Tagged texts saved to ud_vabamorftagger.csv

Assigning NaN values in columns form and pos with an empty string
Removing NaN words
Modified dataframe saved to ud_vabamorftagger.csv
Creating column 'labels'
Column 'labels' created
Modified dataframe saved to ud_vabamorftagger.csv
Creating column 'labels2'
Column 'labels2' created
Modified dataframe saved to ud_vabamorftagger2.csv


In [8]:
display(df_vmt.head(5))

,sentence_id,lemma,words,form,pos,file_prefix,source,labels,labels2
0,0,Mihhailov,Mihhailovile,sg all,H,aja_ee200110,et_edt-ud-train_007.json,sg all_H,sg all_H_Mihhailov
1,0,määra,määrati,ti,V,aja_ee200110,et_edt-ud-train_007.json,ti_V,ti_V_määra
2,0,preemia,preemia,sg g,S,aja_ee200110,et_edt-ud-train_007.json,sg g_S,sg g_S_preemia
3,0,kapitalism,kapitalismi,sg g,S,aja_ee200110,et_edt-ud-train_007.json,sg g_S,sg g_S_kapitalism
4,0,tingimus,tingimustes,pl in,S,aja_ee200110,et_edt-ud-train_007.json,pl in_S,pl in_S_tingimus


In [9]:
print(df_vmt.shape)

(437758, 9)


### VabamorfWithBertTagger(use_postanalysis=True)

In [ ]:
vbtt_file = 'ud_vwbt_true.csv'

if not os.path.exists(vbtt_file):
    create_df_ud_corpus(jsons, ud_dir, 'custom', vbtt_file, use_postanalysis=True)
    
df_vbtt = pd.read_csv(vbtt_file, keep_default_na=False)
clean_df(df_vbtt, vbtt_file)
create_labels_column(df_vbtt, vbtt_file)
create_labels_column2(df_vbtt, vbtt_file.replace(".csv", "2.csv"))

In [8]:
display(df_vbtt.head(5))

,sentence_id,lemma,words,form,pos,file_prefix,source,labels,labels2
0,0,Mihhailov,Mihhailovile,sg all,H,aja_ee200110,et_edt-ud-train_007.json,sg all_H,sg all_H_Mihhailov
1,0,määra,määrati,ti,V,aja_ee200110,et_edt-ud-train_007.json,ti_V,ti_V_määra
2,0,preemia,preemia,sg n,S,aja_ee200110,et_edt-ud-train_007.json,sg n_S,sg n_S_preemia
3,0,kapitalism,kapitalismi,sg g,S,aja_ee200110,et_edt-ud-train_007.json,sg g_S,sg g_S_kapitalism
4,0,tingimus,tingimustes,pl in,S,aja_ee200110,et_edt-ud-train_007.json,pl in_S,pl in_S_tingimus


In [9]:
print(df_vbtt.shape)

(437758, 9)


# Evaluation

In [10]:
df_ud = pd.read_csv('ud_andmestik2.csv', sep=",", encoding="utf-8")
df_vmt = pd.read_csv('ud_vabamorftagger2.csv', sep=",", encoding="utf-8")
#df_vmt2 = pd.read_csv('ud_vabamorf2.csv', sep=",", encoding="utf-8")
#df_vbtf = pd.read_csv('ud_vwbt_false2.csv', sep=",", encoding="utf-8")
df_vbtt = pd.read_csv('ud_vwbt_true2.csv', sep=",", encoding="utf-8")

In [11]:
def group_labels_by_sentence(df):
    # Preparing data for seqeval metrics (needs nested lists)
    grouped = df.groupby(['source', 'sentence_id'])['labels'].apply(list)
    return grouped.reset_index(drop=True).tolist()

def group_labels_by_sentence_lemma(df):
    # Preparing data for seqeval metrics (needs nested lists)
    # label format is form+pos+lemma
    grouped = df.groupby(['source', 'sentence_id'])['labels2'].apply(list)
    return grouped.reset_index(drop=True).tolist()


In [12]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")

### VabamorfTagger evaluation on UD corpus

Metrics are from weighted average

In [13]:
test_df_ud_vabamorf = df_vmt[df_vmt['source'].str.contains('ud-test')].copy()
test_df_gold = df_ud[df_ud['source'].str.contains('ud-test')].copy()

In [ ]:
labels_true = group_labels_by_sentence(test_df_gold)
labels_pred = group_labels_by_sentence(test_df_ud_vabamorf)
results1_1 = poseval.compute(predictions=labels_pred, references=labels_true)

labels_true = group_labels_by_sentence_lemma(test_df_gold)
labels_pred = group_labels_by_sentence_lemma(test_df_ud_vabamorf)
results1_2 = poseval.compute(predictions=labels_pred, references=labels_true)

In [15]:
print("Form+pos:")
print(f"Precision: \t{results1_1['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results1_1['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results1_1['weighted avg']['f1-score']:.4f}")
print()
print("Form+pos+lemma:")
print(f"Precision: \t{results1_2['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results1_2['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results1_2['weighted avg']['f1-score']:.4f}")

Form+pos:
Precision: 	0.9139
Recall: 	0.9067
F1 Score: 	0.9053

Form+pos+lemma:
Precision: 	0.8054
Recall: 	0.8081
F1 Score: 	0.8040


### VabamorfWithBertTagger(use_postanalysis=True) evaluation on UD corpus

Metrics are from weighted average

In [16]:
test_df_ud_vbtt = df_vbtt[df_vbtt['source'].str.contains('ud-test')].copy()
test_df_gold = df_ud[df_ud['source'].str.contains('ud-test')].copy()

In [ ]:
labels_true = group_labels_by_sentence(test_df_gold)
labels_pred = group_labels_by_sentence(test_df_ud_vbtt)
results3_1 = poseval.compute(predictions=labels_pred, references=labels_true)

labels_true = group_labels_by_sentence_lemma(test_df_gold)
labels_pred = group_labels_by_sentence_lemma(test_df_ud_vbtt)
results3_2 = poseval.compute(predictions=labels_pred, references=labels_true)

In [18]:
print("Form+pos:")
print(f"Precision: \t{results3_1['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results3_1['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results3_1['weighted avg']['f1-score']:.4f}")
print()
print("Form+pos+lemma:")
print(f"Precision: \t{results3_2['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results3_2['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results3_2['weighted avg']['f1-score']:.4f}")

Form+pos:
Precision: 	0.9515
Recall: 	0.9473
F1 Score: 	0.9458

Form+pos+lemma:
Precision: 	0.8408
Recall: 	0.8412
F1 Score: 	0.8396


In [19]:
d1 = pd.DataFrame([["VabamorfTagger", "form+pos", results1_1['weighted avg']['precision'], results1_1["weighted avg"]["recall"], results1_1['weighted avg']["f1-score"]],     
                  #["VabamorfWithBertTagger(use_postanalysis=False)", "form+pos", results2_1['weighted avg']["precision"], results2_1["weighted avg"]["recall"], results2_1["weighted avg"]["f1-score"]], 
                  ["VabamorfWithBertTagger(use_postanalysis=True)", "form+pos", results3_1['weighted avg']["precision"], results3_1["weighted avg"]["recall"], results3_1["weighted avg"]["f1-score"]]], 
                 columns = ["model", "label format", "precision", "recall", "f1 score"])
d1

,model,label format,precision,recall,f1 score
0,VabamorfTagger,form+pos,0.913931,0.906679,0.905279
1,VabamorfWithBertTagger(use_postanalysis=True),form+pos,0.951491,0.947311,0.945794


In [20]:
d2 = pd.DataFrame([["VabamorfTagger", "form+pos+lemma", results1_2['weighted avg']['precision'], results1_2["weighted avg"]["recall"], results1_2['weighted avg']["f1-score"]],                   
                  #["VabamorfWithBertTagger(use_postanalysis=False)", "form+pos+lemma", results2_2['weighted avg']["precision"], results2_2["weighted avg"]["recall"], results2_2["weighted avg"]["f1-score"]],                                    
                  ["VabamorfWithBertTagger(use_postanalysis=True)", "form+pos+lemma", results3_2['weighted avg']["precision"], results3_2["weighted avg"]["recall"], results3_2["weighted avg"]["f1-score"]]],                 
                 columns = ["model", "label format", "precision", "recall", "f1 score"])
d2

,model,label format,precision,recall,f1 score
0,VabamorfTagger,form+pos+lemma,0.805377,0.808073,0.803994
1,VabamorfWithBertTagger(use_postanalysis=True),form+pos+lemma,0.840758,0.841210,0.839633


### VabamorfWithBertTagger(use_postanalysis=True) evaluation against VabamorfTagger

In [ ]:
labels_true = group_labels_by_sentence(test_df_ud_vabamorf)
labels_pred = group_labels_by_sentence(test_df_ud_vbtt)
results5_1 = poseval.compute(predictions=labels_pred, references=labels_true)

labels_true = group_labels_by_sentence_lemma(test_df_ud_vabamorf)
labels_pred = group_labels_by_sentence_lemma(test_df_ud_vbtt)
results5_2 = poseval.compute(predictions=labels_pred, references=labels_true)

In [22]:
print("Form+pos:")
print(f"Precision: \t{results5_1['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results5_1['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results5_1['weighted avg']['f1-score']:.4f}")
print()
print("Form+pos+lemma:")
print(f"Precision: \t{results5_2['weighted avg']['precision']:.4f}")
print(f"Recall: \t{results5_2['weighted avg']['recall']:.4f}")
print(f"F1 Score: \t{results5_2['weighted avg']['f1-score']:.4f}")

Form+pos:
Precision: 	0.9492
Recall: 	0.9445
F1 Score: 	0.9456

Form+pos+lemma:
Precision: 	0.9525
Recall: 	0.9413
F1 Score: 	0.9441
